Meant to test how to ingest PDF Files with LangChain

https://www.youtube.com/watch?v=HwzqfXldKlU

https://www.youtube.com/watch?v=2TJxpyO3ei4

https://github.com/pixegami/rag-tutorial-v2

In [ ]:
import pathlib
cwd = pathlib.Path.cwd()
import os
import pandas


chroma_database_dir = cwd / "CDataBase"
# A test file. Any small pdf can be used, this is meant to be a test of the PDFPlumberLoader, not the content of the file itself.
test_file = cwd / "BF01079761.pdf"  # this if an academic paper on Cost Disease in the Arts

# Another test file. This is meant to test vector embeddings, and it doesn't matter if it slips in with the rulebooks.
vector_test_file = cwd / "Kerttu+Lehto+--+Role-Playing+Games+and+Well-Being+--+IJRP+11.pdf"  # this could be used for the basic test as well, it doesn't matter

In [ ]:
import chromadb

from chromadb.config import Settings

from langchain_chroma import Chroma
from langchain_community.document_loaders import PDFPlumberLoader
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama import OllamaLLM
from langchain_ollama import ChatOllama
from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_text_splitters import CharacterTextSplitter
# from langchain_community.vectorstores import Chroma


# from langchain_classic.chains.summarize import load_summarize_chain

In [ ]:
from langchain_core.documents.base import Document as LCDoc

# CHROMA_HOST = "localhost"
# CHROMA_PORT = "8005"
# CHROMA_COLLECTION_NAME = "reports"

# chroma_client = chromadb.HttpClient(host=os.getenv("CHROMA_HOST"), port=int(os.getenv("CHROMA_PORT")), settings=Settings())
# embedding_function = SentenceTransformerEmbeddings(model_name = "qwen3-embedding")
# llm = OllamaLLM(model="phi4")


def get_llm(llm, temp = 1):
    '''
    Returns the local model
    '''
    return ChatOllama(model = llm, temperature = temp)


def get_embeddings():
    '''
    Returns the embedding model
    '''
    return OllamaEmbeddings(model = "qwen3-embedding")


# def add_to_chroma(chunks: list[LCDoc]):
#     '''
#     '''
#     db = Chroma(persist_directory = str(chroma_database_dir), embedding_function = get_embeddings())
#     db.add_documents(new_chunks, ids = new_chunk_ids)
#     db.persist()



In [ ]:
class Document_Loader:
    def __init__(self, chunk_size = 1000, chunk_overlap = 100) -> None:        
        self.chunk_size, self.chunk_overlap = chunk_size, chunk_overlap


    def load_document(self, file_path):
        '''
        '''
        if file_path is None:
            raise ValueError("No document provided")

        if file_path.suffix == ".pdf":
            document = self.load_pdf(file_path)
            
            chunks = self.create_chunks(document)

            self.vectorize(chunks)

    
    def load_pdf(self, file_path):
        '''
        loads a pdf and turns it into a document. Document is a list of LCDoc
        '''
        loader = PDFPlumberLoader(file_path)
        document = loader.load()

        if not document:
            raise ValueError("Could not extract any text from the PDF.")
        
        return loader.load()


    def create_chunks(self, document):
        '''
        Chunks the document
        '''
        split_text = RecursiveCharacterTextSplitter(chunk_size = self.chunk_size, chunk_overlap = self.chunk_overlap, add_start_index = True)

        chunks = split_text.split_documents(document)

        if not chunks:
            raise ValueError("Chunking produced no chunks. Check PDF content.")
        
        return chunks
        
    
    def vectorize(self, chunks):
        '''
        Creates the vectorization of the data
        '''
        vectordb = Chroma.from_documents(documents = chunks, embedding = get_embeddings())
        return vectordb


# def create_chunks(document: list[LCDoc], llm,
#                   chunk_size = 4000, chunk_overlap = 200,
#                   *args, **kwargs) -> list[LCDoc]:
#     '''
#     Splits the text.
#     '''
#     split_text = RecursiveCharacterTextSplitter(chunk_size = chunk_size, chunk_overlap = chunk_overlap)

#     chunks = split_text.split_documents(document)

#     return chunks


# def summary_of_document(chunks, llm, 
#                         chain_type = "map_reduce",
#                         *args, **kwargs):
#     '''
#     Summarizes the document
#     '''
#     # chain_summary = load_summarize_chain(llm, chain_type = chain_type)


# def load_document(file_path: pathlib.Path,
#                   *args, **kwargs) -> list[LCDoc]:
#     '''
#     Imports a document of any type. Must specify which type of document is being imported.
#     '''
#     if file_path is None:
#         raise ValueError("No PDF provided")

#     try:
#         document = document_loader(file_path, *args, **kwargs)
#     except Exception as e:
#         print(f"Error importing document from file\n{file_path}\nError: {type(e)}")
#         return None
    
#     return document



In [ ]:
def RAG_chain(vector_db, answer_style: str, k: int = 5,
              *args, **kwargs):
    '''
    '''
    retriever = vector_db.as_retriever(search_kwargs = {"k": k})

    def join_context(question: str):
        '''
        '''
        docs = retriever.invoke(question)
        if not docs:
            return "No relevant context found in document"
        return "\n\n---\n\n".join(d.page_content for d in docs)
    
    prompt = ChatPromptTemplate.from_template(
        """
        You are a helpful Dungeon Master that answer questions based only on the given rules.

        If the answer is not clearly in the PDF, say:
        "I can't find anything related to your question in the rules."

        User prefers: {answer_style} answers.
        Use the context to answer clearly.

        Context:
        {context}

        Question:
        {question}

        Answer:
        """
        )
    
    llm = get_llm(*args, **kwargs)

    rag_chain = ({"question": RunnablePassthrough(), "context": join_context, "answer_style": lambda _: answer_style} | prompt | llm)

    return rag_chain, retriever




In [ ]:
db = Chroma(persist_directory = str(chroma_database_dir), embedding_function = None)

In [65]:
# test_document: list = load_document(test_file, load_pdf)
# print(type(test_document[0]))

In [66]:
# test_vector_document = load_document(vector_test_file, load_pdf)

In [67]:
# chunks = create_chunks(test_vector_document, llm)

In [68]:
# chunks

In [69]:
print(test_file.suffix)

.pdf
